# Overture — multi-theme download (live)

A single request can pull several themes / types at once; one file is written per feature type. Here we grab places and building footprints for the same block, pinning a release for reproducibility.

## Setup

Pull in the pieces we need: `Catalog` to discover the latest Overture release, the `EarthLens` entry point to run the download, and `geopandas` to inspect the written GeoParquet files.

In [ ]:
import geopandas as gpd
from earthlens.earthlens import EarthLens

from earthlens.overture import Catalog

## Area of interest

Define a small Times Square block as a `[south, north]` / `[west, east]` bounding box, and the directory the output files will be written to.

In [ ]:
LAT_LIM = [40.757, 40.759]  # [south, north]
LON_LIM = [-73.987, -73.984]  # [west, east] — a Times Square block
OUT = "_overture_out"

## Pin a release

Resolve the most recent Overture release id and pin it, so the download is reproducible even after Overture publishes a newer snapshot.

In [ ]:
catalog = Catalog()
release = catalog.latest_release()  # pin for reproducibility
release

## Download both themes

Request `places` and `buildings` in one call. Overture writes one GeoParquet file per feature type, so `download()` returns a list of paths — one per type.

In [ ]:
engine = EarthLens(
    data_source="overture",
    variables={"places": [], "buildings": []},
    aoi=[LON_LIM[0], LAT_LIM[0], LON_LIM[1], LAT_LIM[1]],
    path=OUT,
    release=release,
)
paths = engine.download()
[p.name for p in paths]

## Inspect the results

Read each written file back with `geopandas` and report its feature count and the set of per-row license ids (buildings are OSM-derived ODbL-1.0; places carry permissive licenses).

In [ ]:
for p in paths:
    gdf = gpd.read_parquet(p)
    print(f'{p.name}: {len(gdf)} features, licenses={sorted(gdf.license_id.unique())}')